# Lecture 9: Link Prediction with Node Embeddings (Shallow Embeddings) — Exercises

Companion to the note [Lecture Notes 1-12](../Lecture%20Notes%201-12.md).

Practise shallow node embeddings through the encoder–decoder lens: truncated SVD of the adjacency matrix, Funk-SVD with bias removal and $L_2$ regularisation, Laplacian eigenmaps, DeepWalk / Node2Vec random walks, skip-gram with negative sampling, and the NetMF view that random-walk embeddings are matrix factorisations. You also debug and fix the self-study's Hits@K loop and evaluate embeddings with a proper edge-masking protocol. NumPy only, no networkx or torch.

**18 exercises** · 🧠 Concept ×5 · ✍️ By hand ×6 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

# Zachary's Karate Club (same 0-indexed edge list as networkx.karate_club_graph(), 34 nodes, 78 edges)
KARATE_EDGES = [(0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (0, 10), (0, 11), (0, 12), (0, 13),
    (0, 17), (0, 19), (0, 21), (0, 31), (1, 2), (1, 3), (1, 7), (1, 13), (1, 17), (1, 19), (1, 21), (1, 30), (2, 3),
    (2, 7), (2, 8), (2, 9), (2, 13), (2, 27), (2, 28), (2, 32), (3, 7), (3, 12), (3, 13), (4, 6), (4, 10), (5, 6),
    (5, 10), (5, 16), (6, 16), (8, 30), (8, 32), (8, 33), (9, 33), (13, 33), (14, 32), (14, 33), (15, 32), (15, 33),
    (18, 32), (18, 33), (19, 33), (20, 32), (20, 33), (22, 32), (22, 33), (23, 25), (23, 27), (23, 29), (23, 32),
    (23, 33), (24, 25), (24, 27), (24, 31), (25, 31), (26, 29), (26, 33), (27, 33), (28, 31), (28, 33), (29, 32),
    (29, 33), (30, 32), (30, 33), (31, 32), (31, 33), (32, 33)]
# Ground-truth split after the club broke up: 0 = "Mr. Hi", 1 = "Officer"
KARATE_CLUB = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

def edges_to_adj(edges, n=None, directed=False):
    """Edge list -> dense 0/1 NumPy adjacency matrix."""
    n = n if n is not None else max(max(e) for e in edges) + 1
    A = np.zeros((n, n))
    for u, v in edges:
        A[u, v] = 1
        if not directed:
            A[v, u] = 1
    return A

A_karate = edges_to_adj(KARATE_EDGES, 34)

from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
rng = np.random.default_rng(9)
sigmoid = lambda x: 1 / (1 + np.exp(-x))

def mask_edges(edges, frac=0.2, seed=10):
    """Hide a fraction of edges: returns (training adjacency, list of hidden edges)."""
    r = np.random.default_rng(seed)
    idx = r.choice(len(edges), round(frac * len(edges)), replace=False)
    hidden = [edges[i] for i in idx]
    A_tr = edges_to_adj(edges, 34)
    for u, v in hidden:
        A_tr[u, v] = A_tr[v, u] = 0
    return A_tr, hidden

A_train, hidden_edges = mask_edges(KARATE_EDGES)   # 16 hidden edges, as in the self-study (round(0.2 * 78))

## Part A · Concepts

### Exercise 1 · Encoder, decoder, similarity, loss
*🧠 Concept · ★☆☆*

For each method fill in the four slots of the encoder–decoder framework: $\text{ENC}(v)$, $\text{DEC}(\mathbf z_u,\mathbf z_v)$,
target similarity $\mathbf S[u,v]$, and loss $\ell$.

(a) Truncated SVD of $A$. (b) Laplacian eigenmaps. (c) DeepWalk with negative sampling.
What do all three have in common in the encoder?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

All three are *shallow*: the encoder is a row lookup in a parameter matrix $\mathbf Z$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

| | ENC | DEC | $\mathbf S[u,v]$ | loss |
|---|---|---|---|---|
| SVD | $\mathbf Z[v]$ | $\mathbf z_u^\top\mathbf z_v$ | $A_{uv}$ | squared error $\lVert \mathbf Z\mathbf Z^\top - A\rVert^2$ |
| Laplacian eigenmaps | $\mathbf Z[v]$ | $\lVert\mathbf z_u - \mathbf z_v\rVert^2$ | $A_{uv}$ | $\sum A_{uv}\,\text{DEC}$ (with a normalisation constraint) |
| DeepWalk | $\mathbf Z[v]$ | $\mathbf z_u^\top\mathbf z_v$ (→ $\sigma$ or softmax) | $P_R(v|u)$, co-occurrence in random walks | cross-entropy / negative sampling |

The encoder is always a lookup table $\mathbf Z\in\mathbb R^{|\mathcal V|\times d}$: one free vector per node, nothing shared. The methods differ only
in the decoder, the notion of similarity and the loss.

</details>

### Exercise 2 · Three limitations of shallow embeddings
*🧠 Concept · ★☆☆*

Critical Insight 4.1 lists three bottlenecks of shallow embeddings. For each give a concrete scenario where it hurts:
(1) no parameter sharing, (2) can't use node features, (3) transductive. Which of the three would a new user signing up to a
social network trigger?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think of a 100-million-user platform, of users who have a profile text but few friends, and of what happens at sign-up.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **No sharing**: parameters grow as $O(|\mathcal V|d)$; with $10^8$ users and $d = 128$ that's $1.28\times10^{10}$ floats (~51 GB),
   and a low-degree node gets very few gradient updates, so its vector is poorly estimated.
2. **No features**: a user with a rich profile (text, age, location) but only 1 friend gets an embedding based on that one edge alone;
   the profile is ignored.
3. **Transductive**: a node unseen at training time has no row in $\mathbf Z$. A new sign-up triggers this: you must re-run (part of) the
   optimisation before you can recommend anything. GNNs (Lecture 10) fix all three by computing embeddings from features with shared weights.

</details>

### Exercise 3 · Node2Vec: choosing p and q
*🧠 Concept · ★★☆*

You want embeddings for two different tasks on the same graph:

(a) predicting which **community** a node belongs to (homophily: friends are similar),
(b) finding nodes with the same **structural role** (e.g. bridges or hubs in different parts of the network).

Which walk behaviour (BFS-like or DFS-like) and which $p, q$ settings fit each task? What do you get with $p = q = 1$?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Look at $\alpha_{pq}(t,x)$: $1/p$ weights going back to $t$, $1/q$ weights moving to nodes at distance 2 from $t$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Two views exist, so state which one you use:

* **Lecture view**: a low return parameter $p$ (and large $q$) keeps the walk local, BFS-like, so co-occurring nodes are near each other:
  good for **(a) communities**. A low in-out parameter $q$ drives the walk outward, DFS-like, which the lecture links to **(b) roles across the network**.
* **Original node2vec paper** (Grover & Leskovec): it's the other way round. BFS-like walks ($q>1$) see each node's *immediate* neighbourhood
  in detail, which characterises its local structure, so they capture **structural equivalence (roles)**. DFS-like walks ($q<1$) roam within a
  region and capture **homophily (communities)**.

What's certain either way: $q<1$ pushes the walk outward (DFS-like), $q>1$ keeps it close to $t$ (BFS-like), and $p<1$ makes it backtrack.
In an exam, define BFS/DFS via $\alpha_{pq}$ first, then argue.
$p = q = 1$ gives $\alpha \equiv 1$: the walk is the uniform first-order **DeepWalk** walk.

</details>

### Exercise 4 · Debug the self-study Hits@K loop
*🧠 Concept · ★★☆*

Read the self-study's Hits@K code in the note (section 3.4). It reported a hit rate of $43.75\%$. Find at least **three** problems with
the loop as written. (Hint: there are issues with sort order, the cut-off, which candidates are ranked, and which embedding matrices are
combined.)

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint 1</b></summary>

`np.argsort` sorts in **ascending** order. What sits at position 0?

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

`position < k - 1` for $k=10$ accepts positions 0–8. And should node $u$ itself or its *training* neighbours be in the candidate list?

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **Wrong direction**: `np.argsort` is ascending, so position 0 is the *least* similar node. A hit should be `position >= n - k`, or
   sort by `-score`.
2. **Off by one**: `position < k - 1` gives only the top 9, not the top 10; it should be `< k`.
3. **Candidates not filtered**: the list contains $u$ itself (usually the largest dot product) and all **training** neighbours of $u$, which
   are trivially similar. Standard practice ranks only non-training pairs; otherwise known edges crowd out the hidden one.
4. **Mixed embeddings**: `np.dot(XN[i], X[j])` combines the layout computed on the *masked* graph (`XN`) with the layout on the *full*
   graph (`X`). The full-graph layout has seen the hidden edges, which is test-set leakage.
5. Also: Hits@K ignores *where* in the top $K$ the edge lands; MRR/NDCG are rank-aware (note's brainstorm).
Exercise 13 implements the corrected protocol.

</details>

### Exercise 5 · Why negative sampling?
*🧠 Concept · ★★☆*

Explain why the full softmax loss $\sum_u\sum_{v\in N_R(u)} -\log\frac{\exp(\mathbf z_u^\top\mathbf z_v)}{\sum_{n}\exp(\mathbf z_u^\top\mathbf z_n)}$
is expensive, what negative sampling replaces it with, and why the negatives are drawn from a (smoothed) **unigram/degree** distribution
instead of uniformly.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Count the dot products in the denominator. Think of what a uniformly random node looks like in a scale-free graph.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Every term needs the partition function over **all** $|\mathcal V|$ nodes, so one epoch costs $O(|\mathcal V|^2)$-ish dot products (and gradients
touch every embedding). Negative sampling turns the multi-class problem into $k+1$ binary problems: push $\sigma(\mathbf z_u^\top\mathbf z_v)$
up for the true context and $\sigma(-\mathbf z_u^\top\mathbf z_{n_i})$ up for $k$ sampled "noise" nodes, so the cost is $O(k)$ per pair.
Sampling proportionally to (degree)$^{3/4}$ mimics how often nodes appear as contexts in walks: hubs are frequent contexts, so they must be
frequent negatives too, otherwise the model learns "hubs are similar to everyone". The $3/4$ power slightly boosts rare nodes.

</details>

## Part B · By hand

### Exercise 6 · Laplacian quadratic form
*✍️ By hand · ★★☆*

1. Prove $\sum_{u}\sum_{v} A_{uv}(z_u - z_v)^2 = 2\,\mathbf z^\top L\mathbf z$ with $L = D - A$ for a 1-D embedding $\mathbf z$ (sum over all ordered
   pairs). Generalise to $2\,\mathrm{Tr}(\mathbf Z^\top L\mathbf Z)$ for $d$ dimensions.
2. Path graph $0-1-2$ with $\mathbf z = (1, 0, -1)$: compute the loss both ways.
3. Why must Laplacian eigenmaps add a constraint such as $\mathbf Z^\top\mathbf Z = I$, and why is the eigenvector of eigenvalue 0 discarded?

In [ ]:
lap_loss = None   # value of sum_{u,v} A_uv (z_u - z_v)^2 for the path graph

_A = edges_to_adj([(0, 1), (1, 2)], 3); _z = np.array([1., 0, -1]); _L = np.diag(_A.sum(1)) - _A
check('loss via pairs', lap_loss, sum(_A[u, v] * (_z[u] - _z[v])**2 for u in range(3) for v in range(3)))
check('equals 2 z^T L z', lap_loss, 2 * _z @ _L @ _z)

<details>
<summary><b>💡 Hint</b></summary>

Expand $(z_u - z_v)^2 = z_u^2 - 2z_uz_v + z_v^2$ and use $\sum_v A_{uv} = d_u$ and symmetry of $A$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $\sum_{u,v}A_{uv}(z_u^2 + z_v^2 - 2z_uz_v) = \sum_u d_u z_u^2 + \sum_v d_v z_v^2 - 2\mathbf z^\top A\mathbf z = 2\mathbf z^\top D\mathbf z - 2\mathbf z^\top A\mathbf z = 2\mathbf z^\top L\mathbf z$.
   Applying this to each column of $\mathbf Z$ and summing gives $2\,\mathrm{Tr}(\mathbf Z^\top L\mathbf Z)$.
2. Ordered pairs: $(0,1),(1,0),(1,2),(2,1)$, each contributing $1$, so the loss is $4$. And $L\mathbf z = (1, 0, -1)$, $\mathbf z^\top L\mathbf z = 2$, times 2 $= 4$ ✔.
3. Without a constraint $\mathbf Z = 0$ is optimal (collapse). With $\mathbf Z^\top\mathbf Z = I$ the minimiser is the eigenvectors of the
   **smallest** eigenvalues of $L$ (so this is trace *minimisation*). The eigenvalue-0 eigenvector is the constant vector $\mathbf 1$, which puts
   every node at the same point and carries no information, so we take the next ones (the first is the Fiedler vector).

```python
lap_loss = 4.0
```

</details>

### Exercise 7 · Node2Vec transition probabilities
*✍️ By hand · ★☆☆*

Graph with edges $(0,1), (1,2), (0,2), (1,3), (1,4)$. A Node2Vec walk just moved from $t = 0$ to $v = 1$. With $p = 2$, $q = 0.5$ and unit
weights, compute the normalised probabilities of the next node $x\in N(1) = \{0, 2, 3, 4\}$.

In [ ]:
n2v_probs = None   # [P(x=0), P(x=2), P(x=3), P(x=4)]

# reference from shortest-path distances d(t, x) computed by matrix powers
_A = edges_to_adj([(0, 1), (1, 2), (0, 2), (1, 3), (1, 4)], 5)
_d = np.where(np.eye(5)[0] == 1, 0, np.where(_A[0] > 0, 1, 2))
_a = np.array([{0: 1 / 2, 1: 1.0, 2: 1 / 0.5}[_d[x]] for x in [0, 2, 3, 4]])
check('node2vec probabilities', n2v_probs, _a / _a.sum(), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

$d_{tx}$: $x = 0$ is $t$ itself (0); $x=2$ is a neighbour of $t$ (1); $x = 3, 4$ are two hops from $t$ (2).

</details>

<details>
<summary><b>✅ Solution</b></summary>

Unnormalised weights $\alpha$: $x=0$: $1/p = 0.5$; $x=2$: $1$; $x = 3$: $1/q = 2$; $x=4$: $2$. Sum $= 5.5$.
Probabilities: $(0.0909, 0.1818, 0.3636, 0.3636)$. With $q<1$ the walk prefers moving *away* from $t$ (DFS-like), and $p>1$ discourages
going straight back. DeepWalk would give $0.25$ each.

```python
n2v_probs = np.array([0.5, 1, 2, 2]) / 5.5
```

</details>

### Exercise 8 · One Funk-SVD SGD step
*✍️ By hand · ★★☆*

Observed entry $R_{mu} = 3$, current factors $\mathbf a_m = (1, 0.5)$, $\mathbf b_u = (0.5, 1)$, learning rate $\eta = 0.1$, $\lambda = 0.1$.
Apply the note's update rules once (both updates use the **old** values) and store the new vectors. Did the prediction move towards 3?

In [ ]:
a_new = None
b_new = None
pred_after = None

_a, _b = np.array([1, .5]), np.array([.5, 1]); _e = 3 - _a @ _b
_an = _a + 0.1 * (_e * _b - 0.1 * _a); _bn = _b + 0.1 * (_e * _a - 0.1 * _b)
check('a_m', a_new, _an); check('b_u', b_new, _bn); check('new prediction', pred_after, _an @ _bn)

<details>
<summary><b>💡 Hint</b></summary>

Prediction $= \mathbf a_m^\top\mathbf b_u = 0.5 + 0.5 = 1$, so the error is $e = 2$. Then $\mathbf a \leftarrow \mathbf a + \eta(e\,\mathbf b - \lambda\mathbf a)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$e = 3 - 1 = 2$.
$\mathbf a_m \leftarrow (1,0.5) + 0.1\,[(1, 2) - (0.1, 0.05)] = (1.09, 0.695)$,
$\mathbf b_u \leftarrow (0.5,1) + 0.1\,[(2, 1) - (0.05, 0.1)] = (0.695, 1.09)$.
New prediction $1.09\cdot0.695 + 0.695\cdot1.09 = 1.5151$: closer to 3. The $-\lambda\mathbf a$ term shrinks the vectors slightly every step
(weight decay), which stops low-degree nodes' vectors from exploding.

```python
a_new = np.array([1.09, 0.695])
b_new = np.array([0.695, 1.09])
pred_after = 2 * 1.09 * 0.695
```

</details>

### Exercise 9 · Removing popularity biases
*✍️ By hand · ★☆☆*

Fully observed ratings $R = \begin{pmatrix}5&3&4\\4&2&3\\1&1&1\end{pmatrix}$ (rows $m$, columns $u$). Using the note's bias formula
$b_{mu} = \bar R_m + \bar R_u - \bar R$, compute the baseline $b_{01}$ and the residual $R_{01} - b_{01}$ that Funk-SVD would then factorise.

In [ ]:
baseline_01 = None
residual_01 = None

_R = np.array([[5, 3, 4], [4, 2, 3], [1, 1, 1.]])
_b = _R.mean(1)[0] + _R.mean(0)[1] - _R.mean()
check('baseline', baseline_01, _b, tol=1e-3); check('residual', residual_01, _R[0, 1] - _b, tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Row 0 mean $=4$, column 1 mean $=2$, global mean $= 24/9$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$b_{01} = 4 + 2 - 2.667 = 3.333$, residual $= 3 - 3.333 = -0.333$. Row 0 is a "generous" rater and column 1 an unpopular item; once these
global effects are removed, the residual says this specific pairing is slightly *below* what popularity alone predicts. That's the
personalised signal the inner-product decoder should model (Critical Insight 4.3).

```python
baseline_01 = 4 + 2 - 24 / 9
residual_01 = 3 - baseline_01
```

</details>

### Exercise 10 · Negative-sampling loss by hand
*✍️ By hand · ★☆☆*

For one positive pair with $\mathbf z_u^\top\mathbf z_v = 2$ and $k=2$ negatives with $\mathbf z_u^\top\mathbf z_{n_1} = 1$, $\mathbf z_u^\top\mathbf z_{n_2} = -1$ ($\gamma = 1$),
compute $\mathcal L = -\log\sigma(\mathbf z_u^\top\mathbf z_v) - \sum_i\log\sigma(-\mathbf z_u^\top\mathbf z_{n_i})$. Which term dominates, and what will the gradient do?

In [ ]:
ns_loss = None

check('negative-sampling loss', ns_loss, -np.log(sigmoid(2)) - np.log(sigmoid(-1)) - np.log(sigmoid(1)), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

$-\log\sigma(x) = \log(1 + e^{-x})$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\log(1+e^{-2}) = 0.1269$, $\log(1 + e^{1}) = 1.3133$ (negative $n_1$ with score $+1$), $\log(1+e^{-1}) = 0.3133$. Total $= 1.7535$.
The negative $n_1$ dominates: the model currently thinks $u$ and $n_1$ are similar. The gradient will push $\mathbf z_u$ and $\mathbf z_{n_1}$
apart, and only gently pull $\mathbf z_u, \mathbf z_v$ together (they are already well aligned).

```python
ns_loss = np.log1p(np.exp(-2)) + np.log1p(np.exp(1)) + np.log1p(np.exp(-1))
```

</details>

### Exercise 11 · Counting the cost
*✍️ By hand · ★☆☆*

CORA has $|\mathcal V| = 2708$ nodes. With $d = 64$:

1. How many parameters does a shallow embedding table have?
2. For a single (node, context) pair, how many dot products does the full softmax need, and how many does negative sampling with $k=5$ need?
   What is the speed-up factor?

In [ ]:
n_params = None
speedup = None

check('parameters', n_params, 2708 * 64)
check('speed-up', speedup, 2708 / 6, tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Softmax: one dot product per node in the denominator. Negative sampling: one for the positive plus $k$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $2708\times64 = 173\,312$ parameters, one free vector per node and none shared.
2. Softmax: $2708$ dot products; negative sampling: $1 + 5 = 6$. Speed-up $\approx 451\times$, and it grows linearly with the graph size.

```python
n_params = 2708 * 64
speedup = 2708 / 6
```

</details>

## Part C · Code from scratch

### Exercise 12 · Truncated SVD embeddings and Eckart–Young
*💻 Code · ★☆☆*

Compute the SVD of `A_karate`, keep the top-$k$ singular triplets and build node embeddings $\mathbf Z = U_k\Sigma_k^{1/2}$ (and
$\mathbf Z' = V_k\Sigma_k^{1/2}$ so that $\mathbf Z\mathbf Z'^\top$ is the rank-$k$ reconstruction). Store the Frobenius reconstruction error
for $k = 2, 4, 8, 16$ in `svd_err` (a dict). Eckart–Young says the error is $\sqrt{\sum_{i>k}\sigma_i^2}$; for a symmetric $A$ the singular
values are $|\lambda_i|$.

In [ ]:
def svd_embed(A, k):
    # TODO: return Z, Z_prime
    return None

svd_err = None   # {k: ||A - Z Z'^T||_F}

_lam = np.sort(np.abs(np.linalg.eigvalsh(A_karate)))[::-1]
if svd_err is not None:
    for _k in (2, 4, 8, 16):
        check(f'Eckart-Young error k={_k}', svd_err[_k], np.sqrt(np.sum(_lam[_k:]**2)))
else:
    check('SVD errors', None, 0)

<details>
<summary><b>💡 Hint</b></summary>

`U, s, Vt = np.linalg.svd(A)`; `Z = U[:, :k] * np.sqrt(s[:k])`; `Zp = Vt[:k].T * np.sqrt(s[:k])`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The error falls quickly at first and then slowly; $\lVert A\rVert_F = \sqrt{2m} = \sqrt{156}\approx12.5$. Note that $A$ has negative
eigenvalues, so a *symmetric* factorisation $\mathbf Z\mathbf Z^\top$ (as in the note's loss) can't capture those directions; the
two-factor form $\mathbf Z\mathbf Z'^\top$ can. `torch.svd_lowrank` does the same thing with randomised projections on sparse input.

```python
def svd_embed(A, k):
    U, s, Vt = np.linalg.svd(A)
    return U[:, :k] * np.sqrt(s[:k]), Vt[:k].T * np.sqrt(s[:k])

svd_err = {}
for k in (2, 4, 8, 16):
    Z, Zp = svd_embed(A_karate, k)
    svd_err[k] = np.linalg.norm(A_karate - Z @ Zp.T)
print(svd_err)
```

</details>

### Exercise 13 · A correct ranking evaluation
*💻 Code · ★★☆*

Using the masked graph `A_train` and the 16 `hidden_edges` from the setup:

1. Embed `A_train` with `svd_embed` ($k = 8$) and score all pairs with $S = \mathbf Z\mathbf Z'^\top$; symmetrise as $(S + S^\top)/2$.
2. Implement `rank_metrics(S, A_train, hidden, K=10)` that, for every hidden edge $(u,v)$, ranks **all candidates $w \ne u$ that are not
   training neighbours of $u$** by descending score and finds the rank of $v$ (1 = best). Return `dict(hits=..., mrr=...)`.
3. Also compute `auc_svd`: the AUC of hidden edges vs. all non-edges of the full graph.

In [ ]:
def rank_metrics(S, A_tr, hidden, K=10):
    # TODO
    return None

S_svd = None
metrics_svd = None
auc_svd = None

if metrics_svd is not None:
    _ranks = []
    for u, v in hidden_edges:
        _cand = [w for w in range(34) if w != u and A_train[u, w] == 0]
        _ranks.append(1 + sum(S_svd[u, w] > S_svd[u, v] for w in _cand))
    _ranks = np.array(_ranks)
    check('Hits@10', metrics_svd['hits'], np.mean(_ranks <= 10))
    check('MRR', metrics_svd['mrr'], np.mean(1 / _ranks))
    _iu, _ju = np.triu_indices(34, 1)
    _neg = [(u, v) for u, v in zip(_iu, _ju) if A_karate[u, v] == 0]
    _y = [1] * len(hidden_edges) + [0] * len(_neg)
    check('AUC', auc_svd, roc_auc_score(_y, [S_svd[u, v] for u, v in hidden_edges] + [S_svd[u, v] for u, v in _neg]))
else:
    check('rank metrics', None, 0)

<details>
<summary><b>💡 Hint 1</b></summary>

Mask: `cand = (A_tr[u] == 0) & (np.arange(n) != u)`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Rank of $v$ = 1 + number of candidates with a strictly higher score (optimistic tie handling, which the check uses too).

</details>

<details>
<summary><b>✅ Solution</b></summary>

This is the corrected version of the self-study loop (Exercise 4): descending order, an inclusive top-$K$ cut-off, and only genuine
candidates. The rank-$k$ SVD of the *training* graph smooths the adjacency matrix, so hidden edges inside dense communities get high
reconstructed scores even though they're 0 in `A_train`. Hits@10 and MRR are per-query metrics; the AUC is global (all pairs compete).

```python
def rank_metrics(S, A_tr, hidden, K=10):
    n = len(S)
    ranks = []
    for u, v in hidden:
        cand = (A_tr[u] == 0) & (np.arange(n) != u)
        ranks.append(1 + np.sum(S[u, cand] > S[u, v]))
    ranks = np.array(ranks)
    return dict(hits=np.mean(ranks <= K), mrr=np.mean(1 / ranks))

Z, Zp = svd_embed(A_train, 8)
S_svd = Z @ Zp.T
S_svd = (S_svd + S_svd.T) / 2
metrics_svd = rank_metrics(S_svd, A_train, hidden_edges)
iu, ju = np.triu_indices(34, 1)
pos = np.array([S_svd[u, v] for u, v in hidden_edges])
neg = S_svd[iu, ju][A_karate[iu, ju] == 0]
auc_svd = np.mean((pos[:, None] > neg[None, :]) + 0.5 * (pos[:, None] == neg[None, :]))
print(metrics_svd, 'AUC', auc_svd)
```

</details>

### Exercise 14 · Laplacian eigenmaps and the Fiedler vector
*💻 Code · ★★☆*

1. Build $L = D - A$ for karate, compute its eigen-decomposition (`np.linalg.eigh`) and take the 2-D Laplacian-eigenmap embedding
   `Z_lap` = eigenvectors 2 and 3 (skip the constant one).
2. Verify numerically that $\sum_{u,v}A_{uv}\lVert\mathbf z_u-\mathbf z_v\rVert^2 = 2\,\mathrm{Tr}(\mathbf Z^\top L\mathbf Z)$ for `Z_lap`.
3. Split the nodes by the **sign** of the first column (the Fiedler vector) and store the agreement with the true club split `KARATE_CLUB`
   in `fiedler_acc` (take the better of the two possible label assignments).

In [ ]:
Z_lap = None
lap_pairs = None   # sum_{u,v} A_uv ||z_u - z_v||^2
lap_trace = None   # 2 Tr(Z^T L Z)
fiedler_acc = None

check('pairwise loss = 2 Tr(Z^T L Z)', lap_pairs, lap_trace)
_w = np.linalg.eigvalsh(np.diag(A_karate.sum(1)) - A_karate)
check('loss equals 2*(lambda_2 + lambda_3)', lap_trace, 2 * (_w[1] + _w[2]))
check('Fiedler split recovers the clubs (>= 90%)', None if fiedler_acc is None else fiedler_acc >= 0.9, True)

<details>
<summary><b>💡 Hint</b></summary>

With orthonormal eigenvectors, $\mathrm{Tr}(\mathbf Z^\top L\mathbf Z) = \lambda_2 + \lambda_3$. Accuracy: `max(mean(pred == y), mean(pred != y))`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The pairwise loss equals $2(\lambda_2+\lambda_3)$, the smallest possible value under $\mathbf Z^\top\mathbf Z = I$, $\mathbf Z\perp\mathbf 1$.
The sign of the Fiedler vector alone recovers the real-world split for 32 of 34 members ($94\%$), with no labels used.
This is spectral clustering: minimising $\mathbf z^\top L\mathbf z$ is a relaxation of finding a small cut.

```python
L = np.diag(A_karate.sum(1)) - A_karate
w, V = np.linalg.eigh(L)
Z_lap = V[:, 1:3]
lap_pairs = sum(A_karate[u, v] * np.sum((Z_lap[u] - Z_lap[v])**2) for u in range(34) for v in range(34))
lap_trace = 2 * np.trace(Z_lap.T @ L @ Z_lap)
pred = (Z_lap[:, 0] > 0).astype(int)
fiedler_acc = max(np.mean(pred == KARATE_CLUB), np.mean(pred != KARATE_CLUB))
print(lap_pairs, lap_trace, fiedler_acc)
```

</details>

### Exercise 15 · Funk-SVD matrix completion
*💻 Code · ★★★*

A $40\times30$ "rating" matrix of rank 3 plus noise is given, with only 60% of entries observed (`mask_obs`). Implement Funk-SVD:

1. subtract the bias $\bar R_m + \bar R_u - \bar R$ computed **from observed entries only**;
2. run SGD over the observed entries in random order for 150 epochs with the note's updates ($K = 3$, $\eta = 0.02$, $\lambda = 0.02$);
3. predict all entries (factors + bias) and report the RMSE on the **unobserved** entries in `rmse_funk`. Compare with the bias-only
   baseline `rmse_bias`.

In [ ]:
r_ = np.random.default_rng(0)
R_true = r_.normal(size=(40, 3)) @ r_.normal(size=(3, 30)) + 3
R_obs = R_true + 0.1 * r_.normal(size=R_true.shape)
mask_obs = r_.random(R_true.shape) < 0.6

def funk_svd(R, mask, K=3, eta=0.02, lam=0.02, epochs=150, seed=0):
    # TODO: return the full predicted matrix
    return None

rmse_funk = None
rmse_bias = None

if rmse_funk is not None:
    check('Funk-SVD beats the bias baseline by > 2x', rmse_funk < rmse_bias / 2, True)
    check('Funk-SVD test RMSE < 0.6', rmse_funk < 0.6, True)
else:
    check('Funk-SVD', None, 0)

<details>
<summary><b>💡 Hint 1</b></summary>

Row means over observed entries: `(R * mask).sum(1) / mask.sum(1)`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Loop: `for m, u in obs[r.permutation(len(obs))]: e = Rc[m,u] - A[m] @ B[:, u]; A[m], B[:, u] = A[m] + eta*(e*B[:, u] - lam*A[m]), B[:, u] + eta*(e*A[m] - lam*B[:, u])`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The bias-only model ignores the low-rank interactions (test RMSE ≈ 1.7) while Funk-SVD recovers them from 60% of the entries (test RMSE ≈ 0.43).
Unlike `np.linalg.svd`, the loss only touches observed cells, so missing entries are *predicted* rather than treated as zeros. Try
$\lambda = 0$ with $K = 10$: typically the training error drops further while the test error rises (overfitting on sparse rows, Critical Insight 4.3).

```python
r_ = np.random.default_rng(0)
R_true = r_.normal(size=(40, 3)) @ r_.normal(size=(3, 30)) + 3
R_obs = R_true + 0.1 * r_.normal(size=R_true.shape)
mask_obs = r_.random(R_true.shape) < 0.6

def bias_matrix(R, mask):
    g = R[mask].mean()
    rm = (R * mask).sum(1) / mask.sum(1)
    cm = (R * mask).sum(0) / mask.sum(0)
    return rm[:, None] + cm[None, :] - g

def funk_svd(R, mask, K=3, eta=0.02, lam=0.02, epochs=150, seed=0):
    r = np.random.default_rng(seed)
    bias = bias_matrix(R, mask)
    Rc = R - bias
    A = 0.1 * r.normal(size=(R.shape[0], K)); B = 0.1 * r.normal(size=(K, R.shape[1]))
    obs = np.argwhere(mask)
    for _ in range(epochs):
        for m, u in obs[r.permutation(len(obs))]:
            e = Rc[m, u] - A[m] @ B[:, u]
            A[m], B[:, u] = A[m] + eta * (e * B[:, u] - lam * A[m]), B[:, u] + eta * (e * A[m] - lam * B[:, u])
    return bias + A @ B

R_hat = funk_svd(R_obs, mask_obs)
test = ~mask_obs
rmse_funk = np.sqrt(np.mean((R_hat - R_true)[test]**2))
rmse_bias = np.sqrt(np.mean((bias_matrix(R_obs, mask_obs) - R_true)[test]**2))
print('Funk-SVD', rmse_funk, ' bias only', rmse_bias)
```

</details>

### Exercise 16 · DeepWalk and Node2Vec walkers
*💻 Code · ★★☆*

Implement

* `deepwalk_walk(A, start, length, r)`: uniform first-order walk, $P(v_i|v_{i-1}) = 1/d(v_{i-1})$;
* `node2vec_step(A, t, v, p, q, r)`: one second-order step from $v$ given the previous node $t$, with the bias $\alpha_{pq}(t,x)$.

Validate `node2vec_step` empirically: on the graph of Exercise 7 take 20 000 steps from $(t,v) = (0,1)$ with $p=2, q=0.5$ and store the
empirical frequencies of the next node $x \in (0,2,3,4)$ in `freq_n2v`.

In [ ]:
A_n2v = edges_to_adj([(0, 1), (1, 2), (0, 2), (1, 3), (1, 4)], 5)

def deepwalk_walk(A, start, length, r):
    # TODO: return a list of `length` node ids starting with `start`
    return None

def node2vec_step(A, t, v, p, q, r):
    # TODO: return the next node
    return None

freq_n2v = None

check('node2vec empirical frequencies', freq_n2v, np.array([0.5, 1, 2, 2]) / 5.5, tol=0.015)
_w = deepwalk_walk(A_karate, 0, 20, np.random.default_rng(0))
check('walk has 20 nodes and follows edges', None if _w is None else (len(_w) == 20 and all(A_karate[a, b] == 1 for a, b in zip(_w, _w[1:]))), True)

<details>
<summary><b>💡 Hint 1</b></summary>

$d_{tx}=0$ iff $x = t$; $d_{tx} = 1$ iff $A_{tx} = 1$; otherwise (as $x$ is a neighbour of $v$, itself a neighbour of $t$) $d_{tx}=2$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

`nbrs = np.flatnonzero(A[v]); w = ...; return r.choice(nbrs, p=w / w.sum())`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The empirical frequencies match $(0.091, 0.182, 0.364, 0.364)$ from Exercise 7. Only the previous node $t$ is needed to know the
distances $d_{tx}\in\{0,1,2\}$, which is why Node2Vec is a *second-order* Markov chain. Real implementations precompute alias tables
for every directed edge $(t,v)$ for $O(1)$ sampling.

```python
A_n2v = edges_to_adj([(0, 1), (1, 2), (0, 2), (1, 3), (1, 4)], 5)

def deepwalk_walk(A, start, length, r):
    walk = [start]
    while len(walk) < length:
        walk.append(int(r.choice(np.flatnonzero(A[walk[-1]]))))
    return walk

def node2vec_step(A, t, v, p, q, r):
    nbrs = np.flatnonzero(A[v])
    w = np.array([1 / p if x == t else (1.0 if A[t, x] else 1 / q) for x in nbrs])
    return int(r.choice(nbrs, p=w / w.sum()))

r = np.random.default_rng(1)
steps = np.array([node2vec_step(A_n2v, 0, 1, 2, 0.5, r) for _ in range(20_000)])
freq_n2v = np.array([np.mean(steps == x) for x in (0, 2, 3, 4)])
print(freq_n2v)
```

</details>

### Exercise 17 · DeepWalk: skip-gram with negative sampling
*💻 Code · ★★★*

Train DeepWalk embeddings on karate from scratch:

1. 10 walks of length 20 from every node (`deepwalk_walk`), context window 3 → (centre, context) pairs.
2. Skip-gram with negative sampling: input vectors `Z_in`, output vectors `Z_out` ($d = 8$), $k = 5$ negatives per pair drawn with
   probability $\propto d_v^{3/4}$, minibatches of 256, SGD with $\eta = 0.025$, 5 epochs. Minimise
   $-\log\sigma(\mathbf z_u^\top\mathbf z'_v) - \sum_i\log\sigma(-\mathbf z_u^\top\mathbf z'_{n_i})$.
3. Evaluate: 5-fold cross-validated accuracy of `LogisticRegression` predicting `KARATE_CLUB` from `Z_in`, stored in `dw_acc`.

In [ ]:
def train_deepwalk(A, dim=8, walks_per_node=10, walk_len=20, window=3, k=5, lr=0.025, epochs=5, seed=0):
    r = np.random.default_rng(seed)
    # TODO: return Z_in
    return None

Z_dw = train_deepwalk(A_karate)
dw_acc = None

check('DeepWalk embeddings separate the clubs (CV acc >= 0.8)', None if dw_acc is None else dw_acc >= 0.8, True)

<details>
<summary><b>💡 Hint 1</b></summary>

Gradients for a pair with scores $s_+ = \mathbf z_u^\top\mathbf z'_v$, $s_i = \mathbf z_u^\top\mathbf z'_{n_i}$: $\partial/\partial\mathbf z_u = (\sigma(s_+) - 1)\mathbf z'_v + \sum_i\sigma(s_i)\mathbf z'_{n_i}$, $\partial/\partial\mathbf z'_v = (\sigma(s_+)-1)\mathbf z_u$, $\partial/\partial\mathbf z'_{n_i} = \sigma(s_i)\mathbf z_u$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Use `np.add.at(Z, idx, update)` so repeated indices in a minibatch accumulate correctly.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Even 8-dimensional embeddings learnt purely from walk co-occurrences separate the two factions (CV accuracy around 0.9): nodes that
co-occur in short walks are in the same dense community. Two vectors per node (input/output) is the word2vec convention; using only
`Z_in` as the embedding is standard.

```python
def train_deepwalk(A, dim=8, walks_per_node=10, walk_len=20, window=3, k=5, lr=0.025, epochs=5, seed=0):
    r = np.random.default_rng(seed)
    n = len(A)
    walks = [deepwalk_walk(A, s, walk_len, r) for _ in range(walks_per_node) for s in range(n)]
    pairs = np.array([(w[i], w[j]) for w in walks for i in range(len(w))
                      for j in range(max(0, i - window), min(len(w), i + window + 1)) if i != j])
    noise = A.sum(1)**0.75; noise /= noise.sum()
    Z_in = 0.1 * r.normal(size=(n, dim)); Z_out = 0.1 * r.normal(size=(n, dim))
    for _ in range(epochs):
        r.shuffle(pairs)
        for b in range(0, len(pairs), 256):
            u, v = pairs[b:b + 256].T
            neg = r.choice(n, size=(len(u), k), p=noise)
            zu = Z_in[u]
            g_pos = sigmoid(np.sum(zu * Z_out[v], 1)) - 1
            g_neg = sigmoid(np.einsum('bd,bkd->bk', zu, Z_out[neg]))
            g_u = g_pos[:, None] * Z_out[v] + np.einsum('bk,bkd->bd', g_neg, Z_out[neg])
            np.add.at(Z_out, v, -lr * g_pos[:, None] * zu)
            np.add.at(Z_out, neg, -lr * g_neg[..., None] * zu[:, None, :])
            np.add.at(Z_in, u, -lr * g_u)
    return Z_in

Z_dw = train_deepwalk(A_karate)
dw_acc = cross_val_score(LogisticRegression(), Z_dw, KARATE_CLUB, cv=5).mean()
print('DeepWalk CV accuracy', dw_acc)
```

</details>

### Exercise 18 · NetMF: DeepWalk as matrix factorisation
*💻 Code · ★★★*

Critical Insight 4.2 says DeepWalk implicitly factorises
$\mathbf M = \log\left(\text{vol}(G)\left(\tfrac1T\sum_{r=1}^T\mathbf P^r\right)\mathbf D^{-1}\right) - \log\gamma$, with $\mathbf P = \mathbf D^{-1}\mathbf A$ and
$\text{vol}(G) = \sum_{ij}A_{ij}$.

Compute $\mathbf M$ for karate with $T = 3$, $\gamma = 1$, replacing the element-wise $\log x$ by $\log\max(x, 1)$ (as NetMF does, to avoid
$\log 0$). Take a rank-2 SVD embedding $\mathbf Z = U_2\Sigma_2^{1/2}$ and store the cross-validated club accuracy in `netmf_acc`.
Also check that $\mathbf M$ is symmetric (why is it?).

In [ ]:
M_netmf = None
Z_netmf = None
netmf_acc = None

check('M is symmetric', None if M_netmf is None else np.allclose(M_netmf, M_netmf.T), True)
_d = A_karate.sum(1); _P = A_karate / _d[:, None]
_Mr = np.log(np.maximum(A_karate.sum() * (_P + _P @ _P + _P @ _P @ _P) / 3 / _d[None, :], 1))
check('NetMF matrix', M_netmf, _Mr)
check('NetMF embedding separates clubs (CV acc >= 0.85)', None if netmf_acc is None else netmf_acc >= 0.85, True)

<details>
<summary><b>💡 Hint</b></summary>

$\mathbf P^r\mathbf D^{-1} = \mathbf D^{-1}(\mathbf A\mathbf D^{-1})^{r-1}\mathbf A\mathbf D^{-1}$, a symmetric matrix sandwiched between the same diagonal.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\mathbf P^r\mathbf D^{-1} = (\mathbf D^{-1}\mathbf A)^r\mathbf D^{-1}$ is symmetric because $\mathbf D^{-1}\mathbf A\mathbf D^{-1}\cdots$ reads the same
backwards. Entry $(u,v)$ is the $r$-step walk probability normalised by the degree of $v$: the "pointwise mutual information" of $u, v$
co-occurring in walks. A **deterministic** 2-D factorisation gives a similar club separation to the stochastic SGNS training of Exercise 17
(CV accuracy ≈ 0.94): random-walk objectives and matrix factorisation optimise the same geometry.

```python
d = A_karate.sum(1)
P = A_karate / d[:, None]
T = 3
S_T = sum(np.linalg.matrix_power(P, r) for r in range(1, T + 1)) / T
M_netmf = np.log(np.maximum(A_karate.sum() * S_T / d[None, :], 1)) - np.log(1.0)
U, s, Vt = np.linalg.svd(M_netmf)
Z_netmf = U[:, :2] * np.sqrt(s[:2])
netmf_acc = cross_val_score(LogisticRegression(), Z_netmf, KARATE_CLUB, cv=5).mean()
print('NetMF CV accuracy', netmf_acc)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Lecture Notes 1-12](../Lecture%20Notes%201-12.md).*